In [ ]:
import torch
import pandas as pd
import re
import os
import unicodedata
from unidecode import unidecode
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

In [ ]:
PATH_BASE_TESTES_FINAIS = os.getenv('PATH_BASE_TESTES_FINAIS')
PATH_GABARITO_CORRETOR = os.getenv('PATH_GABARITO_CORRETOR')
PATH_DADOS_ANONIMIZACAO_BERT = os.getenv('PATH_DADOS_ANONIMIZACAO_BERT')

In [ ]:
df_texts = pd.read_csv(PATH_BASE_TESTES_FINAIS)

In [ ]:
def preprocess(relato):
    def remove_stop_words(relato):
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()
df_texts["relato_tratado"] = df_texts["relato"].apply(normalize_text).apply(preprocess)
df_texts

In [ ]:
df_texts["relato_tratado"] = df_texts["relato"].apply(normalize_text).apply(preprocess)
df_texts

In [ ]:
df_texts['relato_tratado'] = df_texts['relato_tratado'].str.lower()

In [ ]:
df_texts

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import pandas as pd

In [ ]:
model_name = "lfcc/bert-portuguese-ner"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
def classify_tokens(text, max_words = 200):
    def split_text(text):
        words = text.split()
        chunks = [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]
        return chunks
        
    classified_tokens = []
    text_chunks = split_text(text)

    for chunk in text_chunks:
        inputs = tokenizer(chunk, return_tensors="pt", add_special_tokens=False)

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=2)

        label_list = model.config.id2label
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        predictions = predictions[0].tolist()

        current_entity = []
        current_entity_type = None

        for token, prediction in zip(tokens, predictions):
            if token.startswith("##"):
                if current_entity:
                    current_entity.append(token[2:])
            else:
                if current_entity:
                    word = ''.join(current_entity)
                    label = 1 if "Pes" in current_entity_type else 0
                    classified_tokens.append({"palavra": word, "y_palavra": label})
                    current_entity = []

                current_entity = [token]
                current_entity_type = label_list[prediction]
                    
        if current_entity:
            token = ''.join(current_entity)
            label = 1 if "Pes" in current_entity_type else 0
            classified_tokens.append({"palavra": token, "y_palavra": label})
    return classified_tokens

In [ ]:
def check_names(output_list : pd.DataFrame):
    gab = pd.read_csv(PATH_GABARITO_CORRETOR)
    copia_output_list = output_list.copy()
    
    copia_output_list = copia_output_list[copia_output_list['y_palavra'] != 2]
    copia_output_list['y_palavra'] = copia_output_list['y_palavra'].astype(int)

    merge_gab_output_list = copia_output_list.merge(gab[['palavra', 'y']], left_on='palavra', right_on='palavra', how='left')
    merge_gab_output_list['y'] = merge_gab_output_list['y'].fillna(0).astype(int)
    merge_gab_output_list = merge_gab_output_list[merge_gab_output_list['y_palavra'] != 2]

    return merge_gab_output_list

In [ ]:
for idx, text in df_texts["relato_tratado"].items():
    output_list = classify_tokens(text)
    output_list = pd.DataFrame(output_list)
    output_list['palavra'] = output_list['palavra'].str.upper()
    pred_gab_names = check_names(output_list)
    df_texts.at[idx, 'bertimbau_qtd_tokens'] = len(output_list['palavra'])
    df_texts.at[idx, 'bertimbau_qtd_pred_nomes'] = len(pred_gab_names[pred_gab_names['y_palavra'] == 1])
    df_texts.at[idx, 'bertimbau_qtd_gab_nomes'] = len(pred_gab_names[pred_gab_names['y'] == 1])
    df_texts.at[idx, 'bertimbau_nomes_acertados'] = len(pred_gab_names[(pred_gab_names['y_palavra'] == 1) & (pred_gab_names['y'] == 1)])
    print(f"{idx} de 230")

In [ ]:
df_texts

In [ ]:
df_texts.to_csv(PATH_DADOS_ANONIMIZACAO_BERT, index=False)